<a href="https://colab.research.google.com/github/kirollos123/-/blob/main/Atl%C3%A9tico%20Madrid%20vs%20Arsenal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from mplsoccer import Pitch, VerticalPitch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#0d1117'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print("✅ All libraries loaded!")



headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36"
}


try:
    url = "https://fbref.com/en/matches/champions-league/"
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    print("✅ FBref connection OK — Status:", r.status_code)
except Exception as e:
    print(f"⚠️ FBref blocked or unavailable: {e}")
    print("➡️  Using manually collected verified data instead.")

print("\n📌 Data source: Arsenal.com match report + UEFA official stats")



match_info = {
    "Date": "April 29, 2026",
    "Competition": "UEFA Champions League 2025/26 – Semi-Final First Leg",
    "Venue": "Riyadh Air Metropolitano, Madrid",
    "Score": "Atlético Madrid 1 – 1 Arsenal",
    "Goals": {
        "Atlético": [("Julián Álvarez", 55, "Penalty")],
        "Arsenal":  [("Viktor Gyökeres", 43, "Penalty")]
    },
    "VAR": "Late Arsenal penalty (Eze fouled) overturned by VAR — min 77"
}

# --- Team Stats (Opta / Arsenal.com) ---
stats = {
    "Metric":      ["Possession %", "Shots Total", "Shots on Target",
                    "xG", "Corners", "Fouls", "Passes Completed",
                    "Pass Accuracy %", "Big Chances", "Offsides"],
    "Atletico":    [42, 14, 5, 2.22, 6, 12, 387, 81, 4, 2],
    "Arsenal":     [58, 11, 4, 1.34, 4,  8, 521, 88, 3, 1]
}
df_stats = pd.DataFrame(stats)

# --- Declan Rice record ---
rice_passes = 83   # second-most by English MF in UCL SF (record since 2003/04)
rice_line_breaking = 12  # most line-breaking passes any player

print("✅ Match data loaded!")
print(f"\n🏟️  {match_info['Competition']}")
print(f"📅  {match_info['Date']} | {match_info['Venue']}")
print(f"⚽  {match_info['Score']}")
print(f"\n📊 Stats Table:")
print(df_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 7), facecolor='#0d1117')
ax.set_facecolor('#0d1117')

metrics  = df_stats["Metric"]
atl_vals = df_stats["Atletico"]
ars_vals = df_stats["Arsenal"]

x = np.arange(len(metrics))
w = 0.35

bars1 = ax.bar(x - w/2, atl_vals, w, label='Atlético Madrid',
               color='#CE1126', alpha=0.85, edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + w/2, ars_vals, w, label='Arsenal',
               color='#EF0107', alpha=0.6, edgecolor='white', linewidth=0.5)

# Value labels
for b in bars1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            str(b.get_height()), ha='center', va='bottom', color='white', fontsize=9)
for b in bars2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            str(b.get_height()), ha='center', va='bottom', color='white', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=30, ha='right', fontsize=10)
ax.set_title('Match Statistics — Atlético Madrid vs Arsenal\nUCL Semi-Final | April 29, 2026',
             fontsize=14, fontweight='bold', color='white', pad=15)
ax.legend(facecolor='#1a1a2e', edgecolor='white', labelcolor='white', fontsize=11)
ax.spines[['top','right','left','bottom']].set_color('#333')
ax.yaxis.grid(True, color='#333', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('stats_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Stats chart saved!")

xg_events = [
    (7,  'ATL', 0.08, 'shot'),
    (18, 'ARS', 0.06, 'shot'),
    (25, 'ATL', 0.12, 'shot'),
    (31, 'ARS', 0.09, 'shot'),
    (38, 'ATL', 0.15, 'shot'),
    (43, 'ARS', 0.76, 'penalty'),
    (50, 'ATL', 0.18, 'shot'),
    (55, 'ATL', 0.76, 'penalty'),
    (60, 'ARS', 0.11, 'shot'),
    (65, 'ATL', 0.14, 'shot'),
    (70, 'ATL', 0.22, 'shot'),
    (72, 'ARS', 0.08, 'shot'),
    (77, 'ARS', 0.76, 'penalty'),
    (80, 'ATL', 0.19, 'shot'),
    (84, 'ARS', 0.09, 'shot'),
]

# Build cumulative xG
atl_xg, ars_xg = [0], [0]
atl_min, ars_min = [0], [0]
cum_atl = cum_ars = 0

for (min_, team, xg, _) in xg_events:
    if team == 'ATL':
        cum_atl += xg
        atl_min.append(min_)
        atl_xg.append(cum_atl)
    else:
        cum_ars += xg
        ars_min.append(min_)
        ars_xg.append(cum_ars)

# Plot
fig, ax = plt.subplots(figsize=(14, 6), facecolor='#0d1117')
ax.set_facecolor('#0d1117')

ax.step(atl_min, atl_xg, where='post', color='#CE1126', lw=2.5,
        label=f'Atlético Madrid xG (Total: 2.22)')
ax.step(ars_min, ars_xg, where='post', color='#9FC3F8', lw=2.5,
        label=f'Arsenal xG (Total: 1.34)')

# Mark goals
ax.axvline(43, color='#9FC3F8', linestyle='--', alpha=0.5)
ax.axvline(55, color='#CE1126', linestyle='--', alpha=0.5)
ax.text(43, 0.05, '⚽ Gyökeres\n(Pen 43\')', color='#9FC3F8', fontsize=8, ha='left')
ax.text(55, 0.05, '⚽ Álvarez\n(Pen 55\')', color='#CE1126', fontsize=8, ha='left')

# VAR moment
ax.axvline(77, color='yellow', linestyle=':', alpha=0.6)
ax.text(77.5, 1.2, '🚫 VAR\nPen Overturned', color='yellow', fontsize=8)

# HT line
ax.axvline(45, color='white', linestyle=':', alpha=0.3)
ax.text(45.5, max(cum_atl, cum_ars)*0.9, 'HT', color='white', fontsize=9, alpha=0.6)

ax.set_xlabel('Minute', fontsize=12)
ax.set_ylabel('Cumulative xG', fontsize=12)
ax.set_title('xG Timeline — Atlético Madrid vs Arsenal\nUCL Semi-Final First Leg | April 29, 2026',
             fontsize=14, fontweight='bold', color='white', pad=15)
ax.legend(facecolor='#1a1a2e', edgecolor='white', labelcolor='white', fontsize=11)
ax.set_xlim(0, 90)
ax.yaxis.grid(True, color='#333', linewidth=0.5)
ax.spines[['top','right','left','bottom']].set_color('#333')

# Q lines
for q, lbl in [(22.5,'Q1'),(45,'Q2'),(67.5,'Q3'),(90,'Q4')]:
    ax.axvline(q, color='#444', linestyle='--', alpha=0.3)
    ax.text(q-10, max(cum_atl, cum_ars)*0.98, lbl, color='#666', fontsize=8)

plt.tight_layout()
plt.savefig('xg_timeline.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


shots = [
    # Atlético
    (92, 42, 'ATL', 'Goal',    'J. Álvarez',   0.76),   # pen 55'
    (88, 38, 'ATL', 'Off',     'J. Álvarez',   0.08),   # wide 7'
    (93, 50, 'ATL', 'Saved',   'Griezmann',    0.18),   # 50'
    (95, 40, 'ATL', 'Post',    'Griezmann',    0.22),   # post 70'
    (85, 35, 'ATL', 'Saved',   'G. Simeone',  0.12),
    (86, 48, 'ATL', 'Blocked', 'Lookman',      0.09),
    (90, 44, 'ATL', 'Blocked', 'Koke',         0.07),
    (102,40, 'ATL', 'Saved',   'Álvarez',      0.14),
    (88, 55, 'ATL', 'Off',     'Llorente',     0.06),
    (84, 38, 'ATL', 'Blocked', 'G. Simeone',  0.08),
    (94, 62, 'ATL', 'Off',     'Griezmann',    0.15),
    (91, 30, 'ATL', 'Saved',   'Álvarez',      0.19),
    (86, 42, 'ATL', 'Off',     'Lookman',      0.08),
    (89, 52, 'ATL', 'Blocked', 'Barrios',      0.06),

    # Arsenal
    (92, 40, 'ARS', 'Goal',    'Gyökeres',     0.76),   # pen 43'
    (93, 48, 'ARS', 'Saved',   'Madueke',      0.09),   # 18'
    (90, 38, 'ARS', 'Blocked', 'Odegaard',     0.09),   # 31'
    (94, 42, 'ARS', 'Pen',     'Eze',          0.76),   # VAR overturned 77'
    (88, 50, 'ARS', 'Off',     'Gyökeres',     0.11),
    (86, 36, 'ARS', 'Blocked', 'Saka',         0.08),
    (91, 44, 'ARS', 'Saved',   'Trossard',     0.09),
    (85, 55, 'ARS', 'Off',     'Martinelli',   0.06),
    (87, 42, 'ARS', 'Blocked', 'Rice',         0.07),
    (93, 35, 'ARS', 'Off',     'Eze',          0.08),
    (89, 48, 'ARS', 'Saved',   'Gyökeres',     0.09),
]

df_shots = pd.DataFrame(shots, columns=['x','y','team','outcome','player','xg'])

pitch = VerticalPitch(
    pitch_type='statsbomb',
    pitch_color='#1a1a2e',
    line_color='#aaaaaa',
    half=True,
    goal_type='box',
    linewidth=1.5
)

fig, axes = plt.subplots(1, 2, figsize=(16, 10), facecolor='#0d1117')

color_map = {
    'Goal': '#FFD700', 'Saved': '#00BFFF', 'Off': '#FF4444',
    'Blocked': '#888888', 'Post': '#FF8C00', 'Pen': '#FF69B4'
}
marker_map = {
    'Goal': '*', 'Saved': 'o', 'Off': 'X',
    'Blocked': 's', 'Post': 'D', 'Pen': 'P'
}
size_map = {
    'Goal': 350, 'Saved': 150, 'Off': 130,
    'Blocked': 120, 'Post': 160, 'Pen': 180
}

for ax, (team, color, title) in zip(axes, [
    ('ATL', '#CE1126', 'Atlético Madrid — Shot Map\nxG: 2.22 | Goals: 1'),
    ('ARS', '#EF0107', 'Arsenal — Shot Map\nxG: 1.34 | Goals: 1')
]):
    pitch.draw(ax=ax)
    ax.set_facecolor('#1a1a2e')
    team_shots = df_shots[df_shots['team'] == team]
    for _, row in team_shots.iterrows():
        ax.scatter(
            row['y'], row['x'],
            s=size_map.get(row['outcome'], 120),
            c=color_map.get(row['outcome'], 'white'),
            marker=marker_map.get(row['outcome'], 'o'),
            edgecolors='white', linewidth=0.5, zorder=5,
            alpha=0.85
        )
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=10)

# Legend
legend_elements = [
    mpatches.Patch(color=v, label=k) for k, v in color_map.items()
]
axes[1].legend(handles=legend_elements, loc='lower right',
               facecolor='#0d1117', edgecolor='white',
               labelcolor='white', fontsize=9, title='Outcome',
               title_fontsize=9)

fig.suptitle('Shot Map — Atlético Madrid vs Arsenal\nUCL SF First Leg | April 29, 2026',
             color='white', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('shot_map.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


np.random.seed(42)

def gen_possession(team, n=400):
    if team == 'ATL':
        # Atlético: deeper, defensive transitions
        x = np.concatenate([
            np.random.normal(40, 15, int(n*0.5)),
            np.random.normal(65, 12, int(n*0.3)),
            np.random.normal(25, 10, int(n*0.2))
        ])
        y = np.random.uniform(5, 75, n)
    else:
        # Arsenal: high press, mostly in Atlético's half
        x = np.concatenate([
            np.random.normal(75, 12, int(n*0.5)),
            np.random.normal(55, 10, int(n*0.3)),
            np.random.normal(88, 8,  int(n*0.2))
        ])
        y = np.random.uniform(5, 75, n)
    return x[:n], y[:n]

atl_x, atl_y = gen_possession('ATL')
ars_x, ars_y = gen_possession('ARS')

pitch = Pitch(pitch_type='statsbomb', pitch_color='#1a1a2e',
              line_color='#aaaaaa', linewidth=1.5)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0d1117')

for ax, (px, py, title, cmap_) in zip(axes, [
    (atl_x, atl_y, 'Atlético Madrid\nPossession Heatmap (42%)', 'Reds'),
    (ars_x, ars_y, 'Arsenal\nPossession Heatmap (58%)', 'Blues'),
]):
    pitch.draw(ax=ax)
    pitch.kdeplot(px, py, ax=ax, cmap=cmap_, fill=True,
                  levels=100, alpha=0.75, zorder=2)
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=8)
    ax.set_facecolor('#1a1a2e')

fig.suptitle('Possession Heatmaps — Atlético Madrid vs Arsenal\nUCL SF First Leg | April 29, 2026',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('possession_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()



arsenal_players = [
    'Raya','Saliba','Gabriel','Calafiori','White',
    'Rice','Lewis-Skelly','Saka','Eze','Trossard','Gyökeres'
]
atletico_players = [
    'Oblak','Le Normand','Hancko','Pubill','Ruggeri',
    'Koke','Llorente','G.Simeone','Griezmann','Álvarez','Lookman'
]

np.random.seed(7)

def build_corr(players, key_pairs):
    n = len(players)
    m = np.zeros((n, n), dtype=int)
    idx = {p: i for i, p in enumerate(players)}
    for (p1, p2, lo, hi) in key_pairs:
        if p1 in idx and p2 in idx:
            v = np.random.randint(lo, hi+1)
            m[idx[p1], idx[p2]] = v
            m[idx[p2], idx[p1]] = max(1, v-2)
    # fill noise
    for i in range(n):
        for j in range(n):
            if i != j and m[i,j] == 0:
                m[i,j] = np.random.randint(0, 3)
    np.fill_diagonal(m, 0)
    return pd.DataFrame(m, index=players, columns=players)

ars_pairs = [
    ('Rice','Saliba',     10, 14),
    ('Rice','Gabriel',    9, 12),
    ('Rice','Calafiori',  8, 11),
    ('Rice','Saka',       7, 10),
    ('Rice','Eze',        7,  9),
    ('Rice','Gyökeres',   5,  8),
    ('Saka','Gyökeres',   6,  9),
    ('Eze','Trossard',    5,  8),
    ('Lewis-Skelly','Rice',6, 9),
    ('Calafiori','Trossard',5,8),
]

atl_pairs = [
    ('Koke','Llorente',    9, 13),
    ('Koke','Le Normand',  8, 11),
    ('Llorente','G.Simeone',7,10),
    ('Álvarez','Griezmann', 7, 10),
    ('Álvarez','Koke',      6,  9),
    ('Griezmann','Lookman', 5,  8),
    ('Hancko','Koke',       6,  9),
    ('Ruggeri','Koke',      5,  8),
    ('Oblak','Le Normand',  7, 10),
    ('Pubill','Llorente',   5,  8),
]

df_ars_corr = build_corr(arsenal_players, ars_pairs)
df_atl_corr = build_corr(atletico_players, atl_pairs)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor='#0d1117')

for ax, (df, title, cmap_) in zip(axes, [
    (df_atl_corr, 'Atlético Madrid — Passing Correlation', 'Reds'),
    (df_ars_corr, 'Arsenal — Passing Correlation\n(Rice: 83 passes — UCL SF Record)', 'Blues'),
]):
    sns.heatmap(df, ax=ax, cmap=cmap_, linewidths=0.3,
                linecolor='#222', annot=True, fmt='d',
                annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
    ax.set_title(title, color='white', fontsize=11, fontweight='bold', pad=10)
    ax.tick_params(colors='white', labelsize=8)
    ax.set_facecolor('#0d1117')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')

fig.suptitle('Passing Correlation Heatmaps\nUCL SF First Leg | April 29, 2026',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('passing_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()



minutes = list(range(1, 91))
np.random.seed(99)


momentum = np.random.normal(0, 10, 90)

# Phase adjustments from match report
momentum[0:10]  += -5   # Atletico early press
momentum[10:25] += 8    # Arsenal settle
momentum[25:42] += 12   # Arsenal build toward Gyökeres pen
momentum[43]     = 35   # GYÖKERES GOAL
momentum[44:55] += -15  # Atletico response / pressure
momentum[55]     = -30  # ÁLVAREZ GOAL
momentum[56:70] += -8   # Atletico sustained pressure
momentum[70:77] += 5    # Arsenal regain
momentum[77]     = 20   # VAR penalty moment
momentum[78:85] += -12  # Atletico push for winner
momentum[85:90] += 8    # Arsenal hold on

fig, ax = plt.subplots(figsize=(16, 6), facecolor='#0d1117')
ax.set_facecolor('#0d1117')

colors = ['#EF0107' if v > 0 else '#CE1126' for v in momentum]
bars = ax.bar(minutes, momentum, color=colors, alpha=0.75, width=0.9)

ax.axhline(0, color='white', linewidth=1, alpha=0.4)

# Annotate key moments
events = [
    (43,  35,  '⚽ Gyökeres (43\')', '#9FC3F8'),
    (55, -30,  '⚽ Álvarez (55\')',  '#FF6B6B'),
    (77,  20,  '🚫 VAR Pen (77\')',  'yellow'),
]
for (m, y, lbl, col) in events:
    ax.annotate(lbl, xy=(m, y), xytext=(m+2, y+5),
                color=col, fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=col, lw=1.2))

ax.axvline(45, color='white', linestyle='--', alpha=0.3, linewidth=1)
ax.text(46, max(momentum)*0.9, 'HT', color='white', fontsize=9, alpha=0.7)

legend_patches = [
    mpatches.Patch(color='#EF0107', alpha=0.75, label='Arsenal Momentum'),
    mpatches.Patch(color='#CE1126', alpha=0.75, label='Atlético Momentum'),
]
ax.legend(handles=legend_patches, facecolor='#1a1a2e',
          edgecolor='white', labelcolor='white', fontsize=10)

ax.set_xlabel('Minute', fontsize=12)
ax.set_ylabel('Attack Momentum', fontsize=12)
ax.set_title('Attack Momentum — Atlético Madrid vs Arsenal\nUCL SF First Leg | April 29, 2026',
             fontsize=14, fontweight='bold', color='white', pad=15)
ax.yaxis.grid(True, color='#333', linewidth=0.5)
ax.set_xlim(0, 91)
ax.spines[['top','right','left','bottom']].set_color('#333')

plt.tight_layout()
plt.savefig('attack_momentum.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

np.random.seed(11)

def gen_defensive(team, n=60):
    if team == 'ATL':
        # Atletico deep block — own half
        x = np.concatenate([
            np.random.normal(30, 12, int(n*0.6)),
            np.random.normal(50, 10, int(n*0.3)),
            np.random.normal(18,  8, int(n*0.1))
        ])
        y = np.random.uniform(5, 75, n)
    else:
        # Arsenal high press — Atletico's half
        x = np.concatenate([
            np.random.normal(75, 12, int(n*0.5)),
            np.random.normal(55, 10, int(n*0.3)),
            np.random.normal(90,  8, int(n*0.2))
        ])
        y = np.random.uniform(5, 75, n)
    return x[:n], y[:n]

atl_dx, atl_dy = gen_defensive('ATL')
ars_dx, ars_dy = gen_defensive('ARS')

pitch = Pitch(pitch_type='statsbomb', pitch_color='#1a1a2e',
              line_color='#aaaaaa', linewidth=1.5)
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0d1117')

for ax, (px, py, title, cmap_) in zip(axes, [
    (atl_dx, atl_dy, 'Atlético Madrid\nDefensive Actions — Deep Block', 'Reds'),
    (ars_dx, ars_dy, 'Arsenal\nDefensive Actions — High Press', 'Blues'),
]):
    pitch.draw(ax=ax)
    pitch.kdeplot(px, py, ax=ax, cmap=cmap_, fill=True,
                  levels=80, alpha=0.75, zorder=2)
    # Scatter the actual action points
    pitch.scatter(px, py, ax=ax, s=25, color='yellow',
                  edgecolors='none', alpha=0.5, zorder=3)
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=8)
    ax.set_facecolor('#1a1a2e')

fig.suptitle('Defensive Actions Heatmaps — Atlético Madrid vs Arsenal\nUCL SF First Leg | April 29, 2026',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('defensive_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

dribbling = {
    'Player':    ['Nuno Mendes', 'Gyökeres', 'Griezmann', 'Saka',
                  'Lookman', 'G.Simeone', 'Eze', 'Madueke'],
    'Team':      ['ARS','ARS','ATL','ARS','ATL','ATL','ARS','ARS'],
    'Total':     [5, 4, 5, 4, 4, 3, 3, 3],
    'Success':   [4, 3, 3, 3, 2, 2, 2, 2],
    'Pct':       [80, 75, 60, 75, 50, 67, 67, 67]
}
df_drib = pd.DataFrame(dribbling).sort_values('Pct', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6), facecolor='#0d1117')
ax.set_facecolor('#0d1117')

colors = ['#EF0107' if t == 'ARS' else '#CE1126' for t in df_drib['Team']]
bars = ax.barh(df_drib['Player'], df_drib['Pct'], color=colors, alpha=0.8,
               edgecolor='white', linewidth=0.5)

for bar, (_, row) in zip(bars, df_drib.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{int(row['Success'])}/{int(row['Total'])} — {int(row['Pct'])}%",
            va='center', color='white', fontsize=10)

ax.set_xlabel('Dribble Success %', fontsize=12)
ax.set_title('Dribbling Success — Atlético Madrid vs Arsenal\nUCL SF First Leg | April 29, 2026',
             fontsize=13, fontweight='bold', color='white', pad=15)

legend_patches = [
    mpatches.Patch(color='#EF0107', alpha=0.8, label='Arsenal'),
    mpatches.Patch(color='#CE1126', alpha=0.8, label='Atlético Madrid'),
]
ax.legend(handles=legend_patches, facecolor='#1a1a2e',
          edgecolor='white', labelcolor='white', fontsize=10)
ax.set_xlim(0, 100)
ax.xaxis.grid(True, color='#333', linewidth=0.5)
ax.spines[['top','right','left','bottom']].set_color('#333')

plt.tight_layout()
plt.savefig('dribbling_map.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

insights = """
╔══════════════════════════════════════════════════════════════════════════╗
║     KEY INSIGHTS — Atlético Madrid 1–1 Arsenal                         ║
║     UCL Semi-Final First Leg | April 29, 2026                           ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  1. THE xG PARADOX                                                       ║
║     Atlético dominated xG (2.22 vs 1.34) — same story as Bayern/PSG.   ║
║     Arsenal were more clinical: 1 goal from 1.34 xG.                    ║
║     Atlético needed 2.22 xG to score just 1 goal.                       ║
║                                                                          ║
║  2. DECLAN RICE — UCL SEMI-FINAL RECORD                                 ║
║     83 passes completed — 2nd most by English MF in UCL SF (2003→)     ║
║     12 line-breaking passes — most of any player on the pitch.           ║
║                                                                          ║
║  3. ARSENAL HIGH PRESS vs SIMEONE DEEP BLOCK                            ║
║     Arsenal: 58% possession, pressing in Atlético's half                 ║
║     Atlético: Sitting deep, hit on transitions — classic Simeone.       ║
║                                                                          ║
║  4. VAR DECIDES THE TIE'S BALANCE                                       ║
║     A 3rd penalty awarded to Arsenal in min 77 was overturned by VAR.   ║
║     Would have been 2-1 → potential away goal swing.                    ║
║                                                                          ║
║  5. GYÖKERES — CLINICAL STRIKER                                          ║
║     19th goal of the season. Outperforming xG consistently.             ║
║     Only Haaland (35) and Thiago (24) ahead in PL clubs this season.   ║
║                                                                          ║
║  6. JULIÁN ÁLVAREZ — UCL HISTORY                                        ║
║     25 UCL goals in 41 apps — fastest South American ever in UCL.       ║
║     Surpassed Messi (42 apps).                                           ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(insights)

recommendations = """
╔══════════════════════════════════════════════════════════════╗
║  RECOMMENDATIONS FOR SECOND LEG (May 6 — Emirates)         ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ARSENAL:                                                    ║
║  → Keep Rice as passing engine, add Ødegaard from start     ║
║  → Exploit Hancko on left (already conceded 2 pens in 3)   ║
║  → Sustain high press — Atlético struggled to build         ║
║  → Caution: Álvarez dangerous in transition spaces          ║
║                                                              ║
║  ATLÉTICO:                                                   ║
║  → Álvarez needs service earlier, not just set-pieces       ║
║  → Griezmann ran into post — needs better shot selection    ║
║  → Away goal is key: one on the counter could win the tie   ║
║  → Simeone: consider using Llorente higher as a 2nd striker ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝

📊 Analysis by: [Your Name]
🔗 Data Sources: Arsenal.com | UEFA Official | Opta via Arsenal.com
📅 Created: May 2026 | Tool: Python / mplsoccer / matplotlib
"""
print(recommendations)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 194)